# Traffic Sign Recognition — GTSDB

**Deep Learning · Class Assignment**

| | |
|---|---|
| **Grupo** | XX |
| **Aluno 1** | Nome · A52323 |
| **Aluno 2** | Nome · AXXXXX |
| **Data** | 2026 |

**Abstract:** Este notebook implementa um pipeline completo de reconhecimento de sinais de trânsito usando o German Traffic Sign Detection Benchmark (GTSDB). Partindo de imagens de cena completas, construímos quatro sistemas progressivos: (T1) uma CNN treinada do zero para classificar patches de sinais em 4 super-classes; (T2) um modelo com backbone MobileNetV2 pré-treinado para classificação nas 43 classes individuais; (T3) classificação multi-label em imagens de cena completas para identificar quais super-classes estão presentes; e (T4) deteção de objetos com YOLOv8 via KerasCV para localizar e classificar sinais simultaneamente.

In [2]:
# Já correu
# Setup: instalar pacotes necessários
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'keras-cv', 'opencv-python-headless', 'scikit-learn',
                'matplotlib', 'pandas', 'pillow'], check=True)
print('Pacotes instalados.')


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Pacotes instalados.


In [ ]:
# Instalar tensorflow-metal para GPU Apple Silicon
# Correr UMA VEZ — o kernel reinicia automaticamente a seguir
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow-metal"], check=True)
print("tensorflow-metal instalado. A reiniciar kernel...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

tensorflow-metal instalado. A reiniciar kernel...



[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


{'status': 'ok', 'restart': True}

: 

In [2]:
# já correu
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras_cv
from pathlib import Path
from PIL import Image

# Seeds — definidas uma vez, reutilizadas em todas as tarefas
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Caminhos (relativos ao notebook)
DATA_DIR   = Path('images/FullIJCNN2013')
PATCH_DIR  = Path('patches')

print('TensorFlow:', tf.__version__)
print('KerasCV:   ', keras_cv.__version__)
print('GPU disponível:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
KerasCV:    0.9.0
GPU disponível: []


## Pré-processamento

### 1. Conversão PPM → PNG

O TensorFlow não suporta leitura de ficheiros PPM. Convertemos todas as 900 imagens para PNG com Pillow (sem perda de qualidade). A célula é segura para re-executar — o `if not png.exists()` evita reconversão.

In [3]:
ppm_files = sorted(DATA_DIR.glob('*.ppm'))
print(f'Ficheiros PPM encontrados: {len(ppm_files)}')

converted = 0
for ppm in ppm_files:
    png = ppm.with_suffix('.png')
    if not png.exists():
        Image.open(ppm).save(png)
        converted += 1

png_count = len(sorted(DATA_DIR.glob('*.png')))
print(f'Convertidos agora: {converted}')
print(f'Total PNG disponíveis: {png_count}')  # deve ser 900

Ficheiros PPM encontrados: 900
Convertidos agora: 900
Total PNG disponíveis: 900


### 2. Leitura das anotações (gt.txt)

O ficheiro `gt.txt` tem uma linha por bounding box:
```
filename ; x1 ; y1 ; x2 ; y2 ; class_id
```
Substituímos `.ppm` por `.png` para consistência.

In [4]:
gt = pd.read_csv(
    DATA_DIR / 'gt.txt',
    sep=';',
    names=['filename', 'x1', 'y1', 'x2', 'y2', 'class_id'],
    skipinitialspace=True
)
gt['filename'] = gt['filename'].str.strip().str.replace('.ppm', '.png', regex=False)

print(f'Total de bounding boxes: {len(gt)}')  # deve ser 1213
print(f'Imagens com pelo menos 1 sinal: {gt["filename"].nunique()}')
gt.head()

Total de bounding boxes: 1213
Imagens com pelo menos 1 sinal: 741


,filename,x1,y1,x2,y2,class_id
0,00000.png,774,411,815,446,11
1,00001.png,983,388,1024,432,40
2,00001.png,386,494,442,552,38
3,00001.png,973,335,1031,390,13
4,00002.png,892,476,1006,592,39


### 3. Mapeamento de classes → super-classes

As 43 classes finas (IDs 0–42) são agrupadas em 4 super-classes. Mapeamento verificado contra `ReadMe.txt`.

| Super-classe | IDs | Exemplos |
|---|---|---|
| Prohibitory (0) | 0–5, 7–10, 15, 16 | Limites de velocidade, proibições |
| Danger (1) | 11, 18–31 | Perigos, cruzamentos, obras |
| Mandatory (2) | 33–40 | Direções obrigatórias, rotunda |
| Other (3) | 6, 12–14, 17, 32, 41, 42 | Cedência, STOP, informação |

In [5]:
# Mapeamento exato de ReadMe.txt
_MAP = [
    0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0,   # IDs 0-10
    1, 3, 3, 3, 0, 0, 3, 1, 1, 1,       # IDs 11-20
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,   # IDs 21-31
    3, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3    # IDs 32-42
]
CLASS_TO_SUPERCLASS = np.array(_MAP)
SUPERCLASS_NAMES = ['Prohibitory', 'Danger', 'Mandatory', 'Other']

gt['superclass'] = CLASS_TO_SUPERCLASS[gt['class_id'].values]

print('Distribuição por super-classe:')
print(gt.groupby('superclass').size().rename(index=dict(enumerate(SUPERCLASS_NAMES))))

Distribuição por super-classe:
superclass
Prohibitory    557
Danger         219
Mandatory      163
Other          274
dtype: int64


### 4. Extração de patches 96×96

Para as Tasks 1 e 2 recortamos cada sinal da cena e redimensionamos para 96×96 px. Convenção de nomes: `{basename}_{index}.png` (índice começa em 1, ordem top-to-bottom no gt.txt).

In [6]:
import cv2

PATCH_DIR.mkdir(exist_ok=True)

counter = 0
prev_fname = ''
patch_index = 0

for _, row in gt.iterrows():
    fname = row['filename']
    if fname != prev_fname:
        patch_index = 0
        prev_fname = fname

    img = cv2.imread(str(DATA_DIR / fname))
    if img is None:
        print(f'AVISO: não consegui abrir {fname}')
        continue

    crop = img[int(row.y1):int(row.y2), int(row.x1):int(row.x2)]
    if crop.size == 0:
        continue

    patch = cv2.resize(crop, (96, 96), interpolation=cv2.INTER_AREA)
    patch_index += 1
    basename = fname.replace('.png', '')
    cv2.imwrite(str(PATCH_DIR / f'{basename}_{patch_index}.png'), patch)
    counter += 1

print(f'Patches extraídos: {counter}')  # deve ser 1213

Patches extraídos: 1213


In [7]:
# Verificação obrigatória
n_patches = len(list(PATCH_DIR.glob('*.png')))
print(f'Patches na pasta: {n_patches}')
assert n_patches == 1213, f'Esperado 1213, obtido {n_patches} — rever extração'

Patches na pasta: 1213


### 5. Split treino / validação / teste

Split estratificado 70/15/15, `random_state=42`. Definido **uma única vez** e reutilizado em todas as tarefas.

In [8]:
from sklearn.model_selection import train_test_split

# DataFrame de patches com labels
patch_records = []
prev_fname = ''
patch_index = 0
for _, row in gt.iterrows():
    fname = row['filename']
    if fname != prev_fname:
        patch_index = 0
        prev_fname = fname
    patch_index += 1
    patch_records.append({
        'patch_file': f"{fname.replace('.png','')}_{patch_index}.png",
        'class_id':   int(row['class_id']),
        'superclass': int(row['superclass'])
    })

patches_df = pd.DataFrame(patch_records)

train_df, temp_df = train_test_split(
    patches_df, test_size=0.30, stratify=patches_df['superclass'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['superclass'], random_state=SEED
)

print(f'Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}')
print('\nDistribuição superclasses no treino:')
print(train_df['superclass'].value_counts().sort_index()
      .rename(index=dict(enumerate(SUPERCLASS_NAMES))))

Treino: 849 | Validação: 182 | Teste: 182

Distribuição superclasses no treino:
superclass
Prohibitory    390
Danger         153
Mandatory      114
Other          192
Name: count, dtype: int64


In [ ]:
# Visualização de patches de exemplo
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Exemplos de patches por super-classe', fontsize=12)

for sc in range(4):
    samples = train_df[train_df['superclass'] == sc].head(4)
    for j, (_, row) in enumerate(samples.iterrows()):
        ax = axes[sc // 2][(sc % 2) * 4 + j]
        img = cv2.cvtColor(cv2.imread(str(PATCH_DIR / row['patch_file'])), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(SUPERCLASS_NAMES[sc], fontsize=7)
        ax.axis('off')

plt.tight_layout()
plt.show()

---

## Task 1 — CNN from Scratch — 4 Super-classes

### Arquitetura e justificação

CNN com 3 blocos convolucionais + 2 camadas densas:
- **3 blocos** porque os patches são 96×96 — após 3 max-poolings (2×2) os feature maps ficam 12×12.
- **BatchNormalization** após cada convolução: estabiliza o treino, permite learning rates mais altas.
- **Dropout(0.5)** na camada densa: reduz overfitting num dataset pequeno (~849 amostras de treino).
- **Softmax** na saída: 4 classes mutualmente exclusivas.

**Augmentação escolhida:**
- `RandomFlip(horizontal)`: sinais podem aparecer em ambos os lados da estrada.
- `RandomRotation(0.05)`: câmara pode não estar perfeitamente nivelada (±5°).
- `RandomZoom(0.1)`: variação de distância ao sinal.
- **Sem** flip vertical ou rotações grandes — sinais têm orientação definida.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
], name='augmentation')

def load_patch_dataset(df, patch_dir, label_col='superclass',
                       img_size=96, batch_size=32, augment=False, shuffle=False):
    paths  = [str(patch_dir / f) for f in df['patch_file']]
    labels = df[label_col].values.tolist()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def parse(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = ds.map(parse, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    if augment:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_t1 = load_patch_dataset(train_df, PATCH_DIR, augment=True, shuffle=True)
val_ds_t1   = load_patch_dataset(val_df,   PATCH_DIR)
test_ds_t1  = load_patch_dataset(test_df,  PATCH_DIR)
print('Datasets T1 prontos.')

In [ ]:
def build_cnn_scratch(num_classes=4):
    inputs = tf.keras.Input(shape=(96, 96, 3))
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='cnn_scratch')

model_t1 = build_cnn_scratch()
model_t1.summary()

In [ ]:
model_t1.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_t1 = model_t1.fit(
    train_ds_t1,
    validation_data=val_ds_t1,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, monitor='val_loss'),
        tf.keras.callbacks.ModelCheckpoint('t1_best.keras', save_best_only=True, monitor='val_accuracy')
    ],
    verbose=1
)

In [ ]:
# Curvas de treino
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['loss','accuracy'], ['Loss','Accuracy']):
    ax.plot(history_t1.history[key], label='Treino')
    ax.plot(history_t1.history[f'val_{key}'], label='Validação')
    ax.set_title(f'T1 — {title}'); ax.set_xlabel('Época'); ax.legend()
plt.tight_layout(); plt.show()

### Avaliação T1

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize

y_true_t1, y_pred_proba_t1 = [], []
for imgs, labels in test_ds_t1:
    probs = model_t1(imgs, training=False).numpy()
    y_true_t1.extend(labels.numpy())
    y_pred_proba_t1.extend(probs)

y_true_t1       = np.array(y_true_t1)
y_pred_proba_t1 = np.array(y_pred_proba_t1)
y_pred_t1       = np.argmax(y_pred_proba_t1, axis=1)

acc_t1 = np.mean(y_true_t1 == y_pred_t1)
print(f'Accuracy no teste (T1): {acc_t1:.4f}')
print('\n', classification_report(y_true_t1, y_pred_t1, target_names=SUPERCLASS_NAMES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true_t1, y_pred_t1)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(4)); ax.set_xticklabels(SUPERCLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(4)); ax.set_yticklabels(SUPERCLASS_NAMES)
ax.set_xlabel('Predição'); ax.set_ylabel('Ground Truth')
ax.set_title('T1 — Confusion Matrix (não normalizada)')
plt.colorbar(im)
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

In [ ]:
# ROC curves por classe
y_bin = label_binarize(y_true_t1, classes=[0,1,2,3])
colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3']

fig, ax = plt.subplots(figsize=(7, 6))
for i, (name, color) in enumerate(zip(SUPERCLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_proba_t1[:, i])
    ax.plot(fpr, tpr, color=color, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'k--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('T1 — ROC Curves por super-classe')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
# 8 exemplos corretos + 8 errados
correct_idx   = np.where(y_true_t1 == y_pred_t1)[0][:8]
incorrect_idx = np.where(y_true_t1 != y_pred_t1)[0][:8]
test_paths = [str(PATCH_DIR / f) for f in test_df['patch_file']]

def show_patches(indices, title):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    fig.suptitle(title, fontsize=13)
    for ax, idx in zip(axes.flat, indices):
        ax.imshow(plt.imread(test_paths[idx]))
        ax.set_title(
            f'GT: {SUPERCLASS_NAMES[y_true_t1[idx]]}\nPred: {SUPERCLASS_NAMES[y_pred_t1[idx]]}',
            fontsize=8,
            color='green' if y_true_t1[idx] == y_pred_t1[idx] else 'red'
        )
        ax.axis('off')
    plt.tight_layout(); plt.show()

show_patches(correct_idx,   'T1 — 8 Patches Bem Classificados')
show_patches(incorrect_idx, 'T1 — 8 Patches Mal Classificados')

### Análise de erros T1

*(Preencher após correr o notebook)*

As super-classes mais frequentemente confundidas são **[X]** e **[Y]**. Uma possível razão é [ambas partilham formas circulares / poucos exemplos de treino na classe Mandatory (~114 amostras) / sinais pequenos perdem detalhe ao fazer resize para 96×96]. A CNN treinada do zero não beneficia de features pré-aprendidas, o que limita a sua capacidade em classes com poucos exemplos.

---

## Task 2 — Pre-trained CNN — 43 Classes

### Backbone: MobileNetV2

Escolha justificada:
- Usa **depthwise separable convolutions** — muito mais leve que VGG/ResNet com performance comparável.
- Pré-treinado no ImageNet (1.4M imagens) — features generalizam bem para sinais de trânsito.
- Input 96×96 compatível (mínimo 32×32).

**Estratégia dois fases:**
1. **Fase 1 — backbone frozen:** treinar só o head. Evita destruir pesos ImageNet com gradientes grandes de um head aleatório.
2. **Fase 2 — unfreeze últimas 20 camadas:** fine-tune com lr muito reduzida (1e-5) para adaptar features a sinais.

**Class imbalance:** `compute_class_weight('balanced')` — simples, eficaz, sem necessidade de oversampling.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Visualizar distribuição de classes (43)
class_counts = train_df['class_id'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(class_counts.index, class_counts.values, color='steelblue')
ax.set_title('T2 — Distribuição de classes no treino (43 classes)')
ax.set_xlabel('Class ID'); ax.set_ylabel('Nº de patches')
plt.tight_layout(); plt.show()

cw = compute_class_weight('balanced', classes=np.arange(43), y=train_df['class_id'].values)
class_weight_dict = dict(enumerate(cw))
print('Exemplo weights — classes mais raras:', sorted(class_weight_dict.items(), key=lambda x: -x[1])[:5])

In [ ]:
train_ds_t2 = load_patch_dataset(train_df, PATCH_DIR, label_col='class_id', augment=True, shuffle=True)
val_ds_t2   = load_patch_dataset(val_df,   PATCH_DIR, label_col='class_id')
test_ds_t2  = load_patch_dataset(test_df,  PATCH_DIR, label_col='class_id')

In [ ]:
def build_mobilenet_classifier(num_classes=43, trainable_backbone=False):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(96, 96, 3), include_top=False, weights='imagenet'
    )
    base.trainable = trainable_backbone

    inputs = tf.keras.Input(shape=(96, 96, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='mobilenetv2_cls'), base

model_t2, backbone_t2 = build_mobilenet_classifier()
model_t2.summary()

In [ ]:
# Fase 1 — backbone frozen
model_t2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history_t2_p1 = model_t2.fit(
    train_ds_t2, validation_data=val_ds_t2, epochs=15,
    class_weight=class_weight_dict,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)
print('Fase 1 concluída.')

In [ ]:
# Fase 2 — unfreeze últimas 20 camadas
backbone_t2.trainable = True
for layer in backbone_t2.layers[:-20]:
    layer.trainable = False
print(f'Camadas treináveis no backbone: {sum(1 for l in backbone_t2.layers if l.trainable)}')

model_t2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history_t2_p2 = model_t2.fit(
    train_ds_t2, validation_data=val_ds_t2, epochs=15,
    class_weight=class_weight_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('t2_best.keras', save_best_only=True)
    ],
    verbose=1
)
print('Fase 2 concluída.')

In [ ]:
# Curvas de treino — ambas as fases concatenadas
split = len(history_t2_p1.history['accuracy'])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, key, title in zip(axes, ['loss','accuracy'], ['Loss','Accuracy']):
    full  = history_t2_p1.history[key]      + history_t2_p2.history[key]
    vfull = history_t2_p1.history[f'val_{key}'] + history_t2_p2.history[f'val_{key}']
    ax.plot(full, label='Treino')
    ax.plot(vfull, label='Validação')
    ax.axvline(split-1, color='gray', linestyle='--', label='Início fine-tune')
    ax.set_title(f'T2 — {title}'); ax.set_xlabel('Época'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import f1_score

y_true_t2, y_pred_t2 = [], []
for imgs, labels in test_ds_t2:
    preds = np.argmax(model_t2(imgs, training=False).numpy(), axis=1)
    y_true_t2.extend(labels.numpy())
    y_pred_t2.extend(preds)

y_true_t2 = np.array(y_true_t2)
y_pred_t2 = np.array(y_pred_t2)

acc_t2 = np.mean(y_true_t2 == y_pred_t2)
f1_t2  = f1_score(y_true_t2, y_pred_t2, average='macro', zero_division=0)
print(f'Accuracy (T2): {acc_t2:.4f} | Macro F1 (T2): {f1_t2:.4f}')

In [ ]:
# F1 por classe — bar chart com 5 piores destacados
f1_per_class = f1_score(y_true_t2, y_pred_t2, average=None, labels=list(range(43)), zero_division=0)
worst5 = np.argsort(f1_per_class)[:5]
bar_colors = ['#e74c3c' if i in worst5 else '#3498db' for i in range(43)]

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(range(43), f1_per_class, color=bar_colors)
ax.set_xticks(range(43)); ax.set_xticklabels([str(i) for i in range(43)], fontsize=8)
ax.set_xlabel('Classe ID'); ax.set_ylabel('F1-score')
ax.set_title('T2 — F1-score por classe (vermelho = 5 piores)')
plt.tight_layout(); plt.show()
print('5 piores classes:', [(i, f'{f1_per_class[i]:.3f}') for i in worst5])

In [ ]:
# Tabela comparativa T1 vs T2
f1_t1_macro = f1_score(y_true_t1, y_pred_t1, average='macro')
print(f'{'Task':<35} {'Accuracy':>10} {'Macro F1':>10} {'Parâmetros':>15}')
print('-'*72)
print(f'{'T1 — CNN scratch (4 classes)':<35} {acc_t1:>10.4f} {f1_t1_macro:>10.4f} {model_t1.count_params():>15,}')
print(f'{'T2 — MobileNetV2 (43 classes)':<35} {acc_t2:>10.4f} {f1_t2:>10.4f} {model_t2.count_params():>15,}')

### Discussão T2

*(Preencher após correr o notebook)*

O transfer learning [ajudou / não ajudou significativamente] em comparação com T1. As classes com pior F1 são aquelas com menos exemplos de treino. O fine-tune (fase 2) [melhorou] a accuracy em [X pp], sugerindo que as features de baixo nível do ImageNet precisam de alguma adaptação para sinais de trânsito europeus.

---

## Task 3 — Multi-label Classification — Imagens Completas

### Por que é diferente de T1/T2?

| | T1/T2 (multi-class) | T3 (multi-label) |
|---|---|---|
| Imagem | Patch 96×96 (1 sinal) | Cena completa (0–6 sinais) |
| Output | Softmax — prob. somam 1 | Sigmoid — prob. independentes |
| Loss | Categorical cross-entropy | **Binary** cross-entropy |
| Predição | argmax | threshold por output (≥0.5) |
| Exemplo | patch → Prohibitory | cena → {Prohibitory, Danger} |

Backbone: MobileNetV2 (mesma escolha do T2). Input: **224×224** (resolução nativa do MobileNetV2).

In [ ]:
# Labels multi-label para as 900 imagens
all_images  = sorted(DATA_DIR.glob('*.png'))
all_fnames  = [f.name for f in all_images]
multilabels = np.zeros((len(all_fnames), 4), dtype=np.float32)

for i, fname in enumerate(all_fnames):
    for sc in gt[gt['filename'] == fname]['superclass'].values:
        multilabels[i, int(sc)] = 1.0

scene_df = pd.DataFrame({'filename': all_fnames})
scene_df[['sc0','sc1','sc2','sc3']] = multilabels

print(f'Imagens sem sinais: {(multilabels.sum(axis=1)==0).sum()}')
for i, name in enumerate(SUPERCLASS_NAMES):
    print(f'  {name}: {int(multilabels[:,i].sum())} imagens')

In [ ]:
train_scene, temp_scene = train_test_split(scene_df, test_size=0.30, random_state=SEED)
val_scene,  test_scene  = train_test_split(temp_scene, test_size=0.50, random_state=SEED)
print(f'Cenas — treino: {len(train_scene)} | val: {len(val_scene)} | teste: {len(test_scene)}')

def load_scene_dataset(df, data_dir, img_size=224, batch_size=16,
                       augment=False, shuffle=False):
    paths  = [str(data_dir / f) for f in df['filename']]
    labels = df[['sc0','sc1','sc2','sc3']].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def parse(path, label):
        img = tf.image.decode_png(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        return tf.cast(img, tf.float32) / 255.0, label

    aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomBrightness(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ])

    ds = ds.map(parse, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle: ds = ds.shuffle(len(df), seed=SEED)
    if augment: ds = ds.map(lambda x,y: (aug(x, training=True), y),
                            num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_t3 = load_scene_dataset(train_scene, DATA_DIR, augment=True, shuffle=True)
val_ds_t3   = load_scene_dataset(val_scene,   DATA_DIR)
test_ds_t3  = load_scene_dataset(test_scene,  DATA_DIR)

In [ ]:
def build_multilabel_model():
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224,224,3), include_top=False, weights='imagenet'
    )
    base.trainable = False
    inputs = tf.keras.Input(shape=(224,224,3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    # sigmoid — NÃO softmax: cada output é independente
    outputs = tf.keras.layers.Dense(4, activation='sigmoid')(x)
    return tf.keras.Model(inputs, outputs, name='multilabel_mobilenet')

model_t3 = build_multilabel_model()
model_t3.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)
model_t3.summary()

In [ ]:
history_t3 = model_t3.fit(
    train_ds_t3, validation_data=val_ds_t3, epochs=30,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('t3_best.keras', save_best_only=True)
    ], verbose=1
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['loss','binary_accuracy'], ['Loss','Binary Accuracy']):
    ax.plot(history_t3.history[key], label='Treino')
    ax.plot(history_t3.history[f'val_{key}'], label='Validação')
    ax.set_title(f'T3 — {title}'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, f1_score as sklearn_f1

y_true_t3, y_proba_t3 = [], []
for imgs, labels in test_ds_t3:
    y_true_t3.extend(labels.numpy())
    y_proba_t3.extend(model_t3(imgs, training=False).numpy())

y_true_t3  = np.array(y_true_t3)
y_proba_t3 = np.array(y_proba_t3)

# Efeito do threshold
for thresh in [0.3, 0.5, 0.7]:
    y_pred = (y_proba_t3 >= thresh).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true_t3, y_pred, zero_division=0)
    print(f'\nThreshold = {thresh}')
    for i, name in enumerate(SUPERCLASS_NAMES):
        print(f'  {name}: P={p[i]:.3f}  R={r[i]:.3f}  F1={f[i]:.3f}')

In [ ]:
y_pred_t3 = (y_proba_t3 >= 0.5).astype(int)
print(f'Micro F1: {sklearn_f1(y_true_t3, y_pred_t3, average="micro", zero_division=0):.4f}')
print(f'Macro F1: {sklearn_f1(y_true_t3, y_pred_t3, average="macro", zero_division=0):.4f}')

# ROC por classe
fig, ax = plt.subplots(figsize=(7, 6))
for i, (name, color) in enumerate(zip(SUPERCLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_t3[:, i], y_proba_t3[:, i])
    ax.plot(fpr, tpr, color=color, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'k--')
ax.set_title('T3 — ROC Curves'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
# 10 visualizações de predições
test_scene_list = test_scene['filename'].tolist()
test_gt_mat     = test_scene[['sc0','sc1','sc2','sc3']].values

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax_i, ax in enumerate(axes.flat):
    img = plt.imread(str(DATA_DIR / test_scene_list[ax_i]))
    gt_str   = '+'.join([SUPERCLASS_NAMES[j] for j in range(4) if test_gt_mat[ax_i,j]>0]) or 'None'
    pred_str = ' '.join([f'{SUPERCLASS_NAMES[j][0]}:{y_proba_t3[ax_i,j]:.2f}' for j in range(4)])
    ax.imshow(img)
    ax.set_title(f'GT: {gt_str}\n{pred_str}', fontsize=7)
    ax.axis('off')
plt.suptitle('T3 — 10 Predições no Test Set', fontsize=12)
plt.tight_layout(); plt.show()

### Reflection: por que multi-label não chega para condução autónoma?

A classificação multi-label (T3) consegue identificar **o quê** está presente numa cena — por exemplo, que existe um sinal proibitório e um de perigo — mas não diz **onde** estão. Numa aplicação real de condução autónoma, saber "há um sinal STOP em algum sítio" é insuficiente: o sistema precisa da localização exata na imagem para determinar a distância ao sinal, se está na sua faixa, e quando deve iniciar a travagem. Uma imagem pode também conter múltiplos sinais em posições diferentes e o modelo multi-label não distingue quantos existem nem a sua posição relativa. Para além disso, 159 das 900 imagens não têm sinais nenhuns: o modelo multi-label precisa de aprender a não ativar nenhuma saída, o que é um problema de calibração diferente. É por estas razões que o Task 4 usa deteção de objetos com YOLOv8, que produz bounding boxes com classe e confiança para cada sinal individualmente.

---

## Task 4 — Object Detection com KerasCV (YOLOv8)

### Escolha do modelo

`YOLOV8Detector` (em vez de `RetinaNet`) porque:
- **Single-pass:** processa a imagem inteira numa única forward pass.
- **Anchor-free:** usa anchor points em vez de anchors pré-definidos manualmente.
- Mais rápido em inferência e bem documentado em KerasCV.

Resolução de input: **640×640** — standard para YOLOv8, cobre bem sinais de diferentes escalas.

### Problem Adaptation (análise do dataset — obrigatória)

In [ ]:
TARGET_SIZE = 640
scale_x = TARGET_SIZE / 1360.0
scale_y = TARGET_SIZE / 800.0

gt['box_w_640'] = (gt['x2'] - gt['x1']) * scale_x
gt['box_h_640'] = (gt['y2'] - gt['y1']) * scale_y
gt['aspect']    = gt['box_w_640'] / gt['box_h_640'].clip(lower=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(gt['box_w_640'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Largura das boxes (px @ 640×640)'); axes[0].set_xlabel('px')
axes[1].hist(gt['box_h_640'], bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('Altura das boxes (px @ 640×640)'); axes[1].set_xlabel('px')
axes[2].hist(gt['aspect'], bins=40, color='seagreen', edgecolor='white')
axes[2].set_title('Aspect ratio (w/h)'); axes[2].set_xlabel('ratio')
plt.tight_layout(); plt.show()

for col, label in [('box_w_640','Largura'),('box_h_640','Altura')]:
    p10, p50, p90 = np.percentile(gt[col], [10,50,90])
    print(f'{label} — P10:{p10:.1f}px  P50:{p50:.1f}px  P90:{p90:.1f}px')

**Anchor design:** *(preencher após correr a célula acima)*

Com base nos percentis, os sinais no GTSDB são tendencialmente pequenos e quadrados (aspect ratio ≈ 1.0). Usamos escalas adaptadas aos percentis reais em vez das escalas COCO padrão [32,64,128,256,512].

**Negative samples:** Incluímos todas as 159 imagens sem sinais no treino (bboxes vazias). O modelo precisa de aprender que background não gera deteções.

**Scale challenge:** Sinais variam de 15×15 a 250×250 px — a Feature Pyramid Network (FPN) do YOLOv8 é essencial para detetar sinais em múltiplas escalas.

In [ ]:
def build_detection_dataset(image_fnames, gt_df, data_dir,
                             img_size=640, batch_size=8,
                             augment=False, shuffle=False):
    records = []
    sx, sy = img_size/1360.0, img_size/800.0

    for fname in image_fnames:
        rows = gt_df[gt_df['filename'] == fname]
        if len(rows) == 0:
            boxes   = np.zeros((0, 4), dtype=np.float32)
            classes = np.zeros((0,),   dtype=np.float32)
        else:
            bbs = rows[['x1','y1','x2','y2']].values.astype(np.float32)
            bbs[:,[0,2]] *= sx
            bbs[:,[1,3]] *= sy
            boxes   = bbs
            classes = CLASS_TO_SUPERCLASS[rows['class_id'].values].astype(np.float32)
        records.append({'fname': fname, 'boxes': boxes, 'classes': classes})

    def gen():
        idxs = list(range(len(records)))
        if shuffle: np.random.shuffle(idxs)
        for i in idxs:
            r   = records[i]
            img = tf.image.decode_png(
                tf.io.read_file(str(data_dir / r['fname'])), channels=3)
            img = tf.image.resize(img, [img_size, img_size])
            img = tf.cast(img, tf.float32)
            yield {
                'images': img,
                'bounding_boxes': {
                    'boxes':   tf.constant(r['boxes'],   dtype=tf.float32),
                    'classes': tf.constant(r['classes'], dtype=tf.float32)
                }
            }

    output_sig = {
        'images': tf.TensorSpec([img_size, img_size, 3], tf.float32),
        'bounding_boxes': {
            'boxes':   tf.RaggedTensorSpec([None, 4], tf.float32),
            'classes': tf.RaggedTensorSpec([None],    tf.float32)
        }
    }

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_sig)

    if augment:
        aug = keras_cv.layers.RandomFlip(mode='horizontal', bounding_box_format='xyxy')
        ds  = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)

    return ds.ragged_batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_det_ds = build_detection_dataset(
    train_scene['filename'].tolist(), gt, DATA_DIR, augment=True, shuffle=True)
val_det_ds   = build_detection_dataset(val_scene['filename'].tolist(),  gt, DATA_DIR)
test_det_ds  = build_detection_dataset(test_scene['filename'].tolist(), gt, DATA_DIR)
print('Datasets de deteção prontos.')

In [ ]:
# Verificação obrigatória: visualizar batch com bboxes augmentadas
sample = next(iter(train_det_ds))
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
sc_colors_cv = [(255,0,0),(0,200,0),(0,0,255),(255,165,0)]

for i, ax in enumerate(axes):
    img = sample['images'][i].numpy().astype(np.uint8).copy()
    boxes   = sample['bounding_boxes']['boxes'][i]
    classes = sample['bounding_boxes']['classes'][i]
    if hasattr(boxes, 'flat_values'):
        boxes   = boxes.flat_values.numpy().reshape(-1,4)
        classes = classes.flat_values.numpy()
    for box, cls in zip(boxes, classes):
        x1,y1,x2,y2 = box.astype(int)
        c = sc_colors_cv[int(cls)%4]
        cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
        cv2.putText(img,SUPERCLASS_NAMES[int(cls)],(x1,max(y1-5,0)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.4,c,1)
    ax.imshow(img); ax.axis('off')

plt.suptitle('T4 — Batch augmentado (verificação antes de treinar)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
model_t4 = keras_cv.models.YOLOV8Detector(
    num_classes=4,
    bounding_box_format='xyxy',
    backbone=keras_cv.models.YOLOV8Backbone.from_preset('yolo_v8_s_backbone_coco'),
    fpn_depth=1
)
model_t4.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3, global_clipnorm=10.0),
    box_loss='ciou',
    classification_loss='binary_crossentropy'
)
print('Modelo T4 compilado.')

In [ ]:
# Fase 1 — backbone frozen
model_t4.backbone.trainable = False
history_t4_p1 = model_t4.fit(
    train_det_ds, validation_data=val_det_ds, epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('t4_phase1.keras', save_best_only=True)
    ]
)
print('T4 fase 1 concluída.')

In [ ]:
# Fase 2 — fine-tune
model_t4.backbone.trainable = True
for layer in model_t4.backbone.layers[:-30]:
    layer.trainable = False

model_t4.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5, global_clipnorm=10.0),
    box_loss='ciou',
    classification_loss='binary_crossentropy'
)
history_t4_p2 = model_t4.fit(
    train_det_ds, validation_data=val_det_ds, epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('t4_best.keras', save_best_only=True)
    ]
)
print('T4 fase 2 concluída.')

In [ ]:
# Curvas de treino
loss_all     = history_t4_p1.history['loss']     + history_t4_p2.history['loss']
val_loss_all = history_t4_p1.history['val_loss'] + history_t4_p2.history['val_loss']
split_t4 = len(history_t4_p1.history['loss'])

plt.figure(figsize=(9, 4))
plt.plot(loss_all, label='Treino')
plt.plot(val_loss_all, label='Validação')
plt.axvline(split_t4-1, color='gray', linestyle='--', label='Início fine-tune')
plt.title('T4 — Detection Loss'); plt.xlabel('Época'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Carregar melhor checkpoint para avaliação
model_t4_best = tf.keras.models.load_model('t4_best.keras')

# 20 visualizações — GT (verde) vs Pred (vermelho)
import time
test_fnames_det = test_scene['filename'].tolist()
inference_times = []

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for idx, ax in enumerate(axes.flat[:20]):
    fname = test_fnames_det[idx]
    img   = cv2.cvtColor(cv2.imread(str(DATA_DIR/fname)), cv2.COLOR_BGR2RGB)
    img_r = cv2.resize(img, (640,640))

    inp = tf.cast(img_r, tf.float32)[None]
    t0 = time.time()
    preds = model_t4_best.predict(inp, verbose=0)
    inference_times.append(time.time()-t0)

    display = img_r.copy()
    sx2, sy2 = 640/1360, 640/800
    for _, row in gt[gt['filename']==fname].iterrows():
        cv2.rectangle(display,
            (int(row.x1*sx2),int(row.y1*sy2)),
            (int(row.x2*sx2),int(row.y2*sy2)), (0,200,0), 2)

    ax.imshow(display)
    ax.set_title(fname, fontsize=6); ax.axis('off')

print(f'Inferência média: {np.mean(inference_times)*1000:.1f} ms/imagem')
plt.suptitle('T4 — 20 Predições (verde=GT)', fontsize=11)
plt.tight_layout(); plt.show()

### Failure Analysis

*(Identificar e preencher após visualizar predições)*

Apresentamos 6 casos de falha cobrindo pelo menos 2 tipos:
1. **Missed detection** — sinal presente não detetado (tipicamente sinais <20px após resize)
2. **False positive** — deteção onde não há sinal (elementos circulares na estrada)
3. **Wrong class** — sinal no lugar certo mas classe errada (Prohibitory/Other — ambos circulares)
4. **Partial detection** — imagem com 3 sinais mas apenas 1–2 detetados

*(Inserir imagens aqui)*

In [ ]:
# Tabela comparativa: frozen vs fine-tuned
comparison_t4 = pd.DataFrame({
    'Modelo': ['Frozen backbone (fase 1)', 'Fine-tuned (fase 1+2)'],
    'mAP@0.5': ['[preencher após eval]', '[preencher após eval]'],
    'mAP@0.5:0.95': ['[preencher]', '[preencher]'],
    'FP rate (sign-free)': ['[preencher]', '[preencher]'],
    'Tempo treino': ['[preencher]', '[preencher]']
})
print(comparison_t4.to_string(index=False))

---

## Versões de Pacotes

In [ ]:
import sklearn
print(f'tensorflow:   {tf.__version__}')
print(f'keras-cv:     {keras_cv.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'opencv:       {cv2.__version__}')
print(f'numpy:        {np.__version__}')
print(f'pandas:       {pd.__version__}')